# Embedding test

- text-embedding-ada-002
- text-embedding-3-small (1536)
- text-embedding-3-small (512)

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## Ingest

In [3]:
import os
from openai import AzureOpenAI

client = AzureOpenAI(
  api_key = os.environ['JP_AOAI_KEY'],
  api_version = "2023-05-15",
  azure_endpoint = os.environ['JP_AOAI_ENDPOINT']
)

MODEL_ADA_002 = "text-embedding-ada-002"
MODEL_3_SMALL = "text-embedding-3-small"
def generate_embeddings(text):
    response = client.embeddings.create(
        input = text,
        model=MODEL_ADA_002
    )
    return response.data[0].embedding

client_eastus = AzureOpenAI(
  api_key = os.environ['EU_AOAI_KEY'],
  api_version = "2023-05-15",
  azure_endpoint = os.environ['EU_AOAI_ENDPOINT']
)

def generate_embeddings_3(text, dim=1536):
    response = client_eastus.embeddings.create(
        input = text,
        model=MODEL_3_SMALL,
        dimensions=dim
    )
    return response.data[0].embedding

## Sample data

- vector: text-embedding-ada-002 (1536)
- vector3: text-embedding-3-small (1536)
- vector3s: text-embedding-3-small (512)

In [5]:
# read data
import pandas as pd
df = pd.read_pickle('./aoai-docs-vector3.pkl')
df

,title,chunk,parent_id,chunk_id,vector,vector3,vector3s
0,Customize a model with Azure OpenAI Service an...,## Prerequisites\n\n- Read the [When to use Az...,fine-tuning-python.md,0,"[-0.0015928514767438173, -0.010974161326885223...","[-0.021617770195007324, 0.0402211993932724, 0....","[-0.03213920071721077, 0.05979697406291962, 0...."
1,Customize a model with Azure OpenAI Service an...,The training and validation data you use **mus...,fine-tuning-python.md,1,"[-0.023874465376138687, 0.010888501070439816, ...","[0.007704917341470718, 0.030702751129865646, 0...","[0.011283145286142826, 0.044961366802453995, 0..."
2,Customize a model with Azure OpenAI Service an...,"```json\n{""prompt"": ""<prompt text>"", ""completi...",fine-tuning-python.md,2,"[-0.017334766685962677, -0.004344400949776173,...","[-0.0012759960955008864, 0.02082199975848198, ...","[-0.0018883325392380357, 0.03081424906849861, ..."
3,Customize a model with Azure OpenAI Service an...,# [OpenAI Python 0.28.1](#tab/python)\n\n```py...,fine-tuning-python.md,3,"[-0.021893125027418137, -0.018416300415992737,...","[-0.03403666242957115, 0.02480599284172058, 0....","[-0.049646373838186264, 0.03618238493800163, 0..."
4,Customize a model with Azure OpenAI Service an...,job_id = response.id\n\n# You can use the job ...,fine-tuning-python.md,4,"[-0.0271502323448658, -0.014661675319075584, 0...","[-0.03774658218026161, 0.031171677634119987, 0...","[-0.05706236883997917, 0.047122932970523834, 0..."
...,...,...,...,...,...,...,...
435,How to create Assistants with Azure OpenAI Ser...,"""assistant_id"": ""asst_eHwhP4Xnad0bZdJrjHO2hfB4...",assistant.md,10,"[-0.03664305806159973, 0.008231116458773613, 0...","[-0.02476349286735058, 0.00833137333393097, -0...","[-0.03601253777742386, 0.012115975841879845, -..."
436,How to create Assistants with Azure OpenAI Ser...,Extract the new image file ID and download and...,assistant.md,11,"[-0.03731171786785126, -0.0038195045199245214,...","[0.011932477355003357, 0.03829050436615944, -0...","[0.017229046672582626, 0.05528682842850685, -0..."
437,How to create Assistants with Azure OpenAI Ser...,# Gather citations based on annotation attribu...,assistant.md,12,"[-0.036548104137182236, 0.019296955317258835, ...","[-0.014187337830662727, 0.03972454369068146, -...","[-0.02082144282758236, 0.05830003693699837, -0..."
438,Quickstart: Use the OpenAI Service to make you...,## Prerequisites\n\n- An Azure subscription - ...,rest.md,0,"[-0.0027009497862309217, -0.001067467732354998...","[-0.005088833160698414, 0.018993057310581207, ...","[-0.007373328786343336, 0.027519481256604195, ..."


In [6]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(A, B):
    return np.dot(A, B) / (norm(A) * norm(B))

## Test

In [16]:
def embeddding_test(query, model="ada-002"):
    results = None

    if (model == "3-small-512"):
        vector = generate_embeddings_3(query, 512)

        df["cosine_sim"] = df['vector3s'].apply(lambda x: cosine_similarity(x, vector))
        results = df.sort_values("cosine_sim", ascending=False).head(5)    
    elif (model == "3-small"):
        vector = generate_embeddings_3(query)

        df["cosine_sim"] = df['vector3'].apply(lambda x: cosine_similarity(x, vector))
        results = df.sort_values("cosine_sim", ascending=False).head(5)
    else: #"ada-002"
        vector = generate_embeddings(query)

        df["cosine_sim"] = df['vector'].apply(lambda x: cosine_similarity(x, vector))
        results = df.sort_values("cosine_sim", ascending=False).head(5)

    for i in range(len(results)):
        r = results.iloc[i]
        print(f"{r['title']} / {r['parent_id']}({r['chunk_id']}) : {r['cosine_sim']:.4f}")

### ada-002

In [42]:
query = "azure region availability for gpt-3.5-turbo 0125"
embeddding_test(query)

Azure OpenAI Service models / models.md(3) : 0.8422
Whats new in Azure OpenAI Service? / whats-new.md(0) : 0.8348
Whats new in Azure OpenAI Service? / whats-new.md(3) : 0.8340
Whats new in Azure OpenAI Service? / whats-new.md(4) : 0.8294
Azure OpenAI Service models / models.md(2) : 0.8204


### 3-small (1536)

In [43]:
embeddding_test(query, '3-small')

Azure OpenAI Service models / models.md(3) : 0.5949
Azure OpenAI Service models / models.md(2) : 0.5706
Whats new in Azure OpenAI Service? / whats-new.md(0) : 0.5468
Whats new in Azure OpenAI Service? / whats-new.md(4) : 0.5340
Azure OpenAI Service legacy models / legacy-models.md(0) : 0.5280


### 3-small (512)

In [44]:
embeddding_test(query, '3-small-512')

Azure OpenAI Service models / models.md(3) : 0.6162
Azure OpenAI Service models / models.md(2) : 0.5954
Azure OpenAI Service legacy models / legacy-models.md(0) : 0.5448
Using your data with Azure OpenAI Service / use-your-data.md(9) : 0.5324
Whats new in Azure OpenAI Service? / whats-new.md(0) : 0.5303


In [41]:
row = df[(df['parent_id'] == 'models.md') & (df['chunk_id'] == 3)]
print(row['chunk'].iloc[0])

#### Azure Government regions

The following GPT-4 models are available with [Azure Government](/azure/azure-government/documentation-government-welcome):

|Model ID | Model Availability |
|--|--|
| `gpt-4` (1106-preview) | US Gov Virginia<br>US Gov Arizona |


### GPT-3.5 models

> [!IMPORTANT]
> The NEW `gpt-35-turbo (0125)`  model has various improvements, including higher accuracy at responding in requested formats and a fix for a bug which caused a text encoding issue for non-English language function calls.

GPT-3.5 Turbo is used with the Chat Completion API. GPT-3.5 Turbo version 0301 can also be used with the Completions API.  GPT-3.5 Turbo versions 0613 and 1106 only support the Chat Completions API.

GPT-3.5 Turbo version 0301 is the first version of the model released.  Version 0613 is the second version of the model and adds function calling support.

See [model versions](../concepts/model-versions.md) to learn about how Azure OpenAI Service handles model version upgrades, 

Performance note([MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard)):
- 3-large - top
- 3-large-512 - 2nd
- 3-small - 3rd
- ada-002 - 4th

=> ada-002 - more region availability (latency)

Reference:
https://community.openai.com/t/transitioning-to-the-new-embeddings-models-from-ada/602761/7